# Whisper large-v3 LoRA fine-tune — Greek council ASR (dry run)

This is a dry run. The point is to check the whole pipeline holds together and to get an early read on the ~1.9k curated corrections (about 1.8 hours of audio). It is not evidence that the model got better — there is far too little data here, so expect it to overfit.

To run it on Kaggle: set the Accelerator to GPU T4 x2, turn Internet on, then use Save & Run All (Commit) for a headless run.

One thing on memory: each meeting's mp3 is decoded to PCM one at a time and freed before the next one loads. An earlier version kept every meeting in RAM at once and ran out of memory.

In [ ]:
# 1. Install
# NB: Kaggle ships torchao==0.10.0, which PEFT's LoRA dispatcher rejects with an
# ImportError ("only versions above 0.16.0 are supported"). We don't use torchao
# (plain fp16 LoRA), and is_torchao_available() returns False cleanly when it's
# absent — so just remove it.
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install -U transformers datasets peft accelerate evaluate jiwer faster-whisper librosa soundfile 2>/dev/null
import torch; print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# 2. Config
EXPORT_URL = "https://79-76-114-184.sslip.io/api/export"
MEETING_API = "https://opencouncil.gr/api/cities/{city}/meetings/{meeting}"
MODEL_ID   = "openai/whisper-large-v3"
LANGUAGE, TASK = "greek", "transcribe"
VAL_CITIES = {"orestiada", "argos"}
SMOKE      = True                 # True = fast smoke test; set False for the full run
SAMPLE_N   = 60 if SMOKE else None
# Smoke speed is dominated by the number of DISTINCT meetings whose full mp3 we
# download+decode (each council meeting is hours of audio). Capping clips alone
# (SAMPLE_N) doesn't help if those clips still span ~84 meetings — cap meetings.
SMOKE_TRAIN_MEETINGS = 4 if SMOKE else None
SMOKE_VAL_MEETINGS   = 2 if SMOKE else None
VAL_REG_PER_MEETING = 8
SR = 16000; PAD_S = 0.2; MIN_DUR, MAX_DUR = 0.3, 30.0; SEED = 13
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR, TRAIN_BS, GRAD_ACC, EVAL_BS = 1e-4, 2, 4, 4
EPOCHS = 1 if SMOKE else 4
OUT_DIR = "/kaggle/working/whisper-lora-greek"
import os, random, numpy as np
os.makedirs(OUT_DIR, exist_ok=True); random.seed(SEED); np.random.seed(SEED)

In [ ]:
# 3. Fetch export + meeting JSON helper
import requests, json, collections
def fetch_jsonl(url):
    r = requests.get(url, timeout=600); r.raise_for_status()
    return [json.loads(l) for l in r.text.splitlines() if l.strip()]
rows = fetch_jsonl(EXPORT_URL)
print("included rows:", len(rows))
print("per city:", dict(collections.Counter(r["city_id"] for r in rows).most_common()))
def fetch_meeting(city, meeting):
    r = requests.get(MEETING_API.format(city=city, meeting=meeting),
                     headers={"User-Agent":"oc-ft/1.0","Accept":"application/json"}, timeout=120)
    r.raise_for_status(); return r.json()

In [ ]:
# 4. Audio helpers (NO global PCM cache — decode per meeting, free after)
import librosa, hashlib, pathlib, gc
CACHE = pathlib.Path("/tmp/audio_cache"); CACHE.mkdir(parents=True, exist_ok=True)
def dl(url):
    p = CACHE / (hashlib.md5(url.encode()).hexdigest() + ".mp3")
    if not p.exists():
        with requests.get(url, stream=True, timeout=600) as r:
            r.raise_for_status()
            with open(p, "wb") as f:
                for c in r.iter_content(1 << 20): f.write(c)
    return str(p)
def load_pcm_path(path): return librosa.load(path, sr=SR, mono=True)[0]
def cut(y, start, end):
    a = max(0, int((start - PAD_S) * SR)); b = min(len(y), int((end + PAD_S) * SR)); return y[a:b]
def ok_span(s, e):
    d = (e or 0) - (s or 0); return MIN_DUR <= d <= MAX_DUR

In [ ]:
# 5. Memory-safe build: slice clips and STREAM to disk (wav). RAM holds only a manifest.
import gc, soundfile as sf
CLIPS = pathlib.Path("/tmp/clips"); CLIPS.mkdir(parents=True, exist_ok=True)
train_src = [r for r in rows if r["city_id"] not in VAL_CITIES and r.get("final_after_text") and ok_span(r["start"], r["end"])]
val_src   = [r for r in rows if r["city_id"] in VAL_CITIES and r.get("final_after_text") and ok_span(r["start"], r["end"])]

def cap_meetings(src, n):
    # Keep only rows from the first n randomly-chosen meetings -> fewer full-mp3 downloads.
    if not n: return src
    mids = list({(r["city_id"], r["meeting_id"]) for r in src})
    random.shuffle(mids); keep = set(mids[:n])
    return [r for r in src if (r["city_id"], r["meeting_id"]) in keep]

train_src = cap_meetings(train_src, SMOKE_TRAIN_MEETINGS)
val_src   = cap_meetings(val_src, SMOKE_VAL_MEETINGS)
if SAMPLE_N: random.shuffle(train_src); train_src = train_src[:SAMPLE_N]
print(f"sources: train_src={len(train_src)} ({len({(r['city_id'],r['meeting_id']) for r in train_src})} mtgs) "
      f"val_src={len(val_src)} ({len({(r['city_id'],r['meeting_id']) for r in val_src})} mtgs)")

reg_src = []
for city, mtg in sorted({(r["city_id"], r["meeting_id"]) for r in val_src}):
    try: mj = fetch_meeting(city, mtg)
    except Exception as e: print("skip", city, mtg, e); continue
    au = (mj.get("meeting") or {}).get("audioUrl")
    if not au: continue
    ne = [u for seg in (mj.get("transcript") or []) for u in (seg.get("utterances") or [])
          if u.get("lastModifiedBy") is None and ok_span(u.get("startTimestamp"), u.get("endTimestamp")) and (u.get("text") or "").strip()]
    random.shuffle(ne)
    for u in ne[:VAL_REG_PER_MEETING]:
        reg_src.append({"audio_url": au, "start": u["startTimestamp"], "end": u["endTimestamp"], "text": u["text"]})

def mk(r, dst, k): return {"url": r["audio_url"], "start": r["start"], "end": r["end"], "text": r[k], "dst": dst}
tasks = ([mk(r,"train","final_after_text") for r in train_src]
       + [mk(r,"valc","final_after_text") for r in val_src]
       + [mk(r,"valr","text") for r in reg_src])
buckets = collections.defaultdict(list)
for t in tasks: buckets[t["url"]].append(t)

man = {"train": [], "valc": [], "valr": []}
fail = 0; idx = 0
for i, (url, ts) in enumerate(buckets.items(), 1):
    try:
        path = dl(url); y = load_pcm_path(path)        # ONE meeting in RAM at a time
    except Exception as e:
        fail += len(ts); print("audio fail", url, str(e)[:60]); continue
    for t in ts:
        wav = str(CLIPS / f"{t['dst']}_{idx}.wav"); idx += 1
        sf.write(wav, cut(y, t["start"], t["end"]), SR)   # stream clip to disk
        man[t["dst"]].append({"audio": wav, "text": (t["text"] or "").strip()})
    del y; gc.collect()                                # free RAM + disk before next meeting
    try: os.remove(path)
    except Exception: pass
    if i % 20 == 0: print(f"{i}/{len(buckets)} meetings | train={len(man['train'])} valc={len(man['valc'])} valr={len(man['valr'])}")
print(f"DONE train={len(man['train'])} valc={len(man['valc'])} valr={len(man['valr'])} audio_fail={fail}")

In [ ]:
# 6. HF datasets (LAZY audio from disk) + Whisper preprocessing -> Arrow on disk
from datasets import Dataset, Audio
from transformers import WhisperProcessor
processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
def to_ds(recs):
    if not recs: return None
    d = Dataset.from_list(recs).cast_column("audio", Audio(sampling_rate=SR))
    def prep(b):
        a = b["audio"]
        b["input_features"] = processor.feature_extractor(a["array"], sampling_rate=a["sampling_rate"]).input_features[0]
        b["labels"] = processor.tokenizer(b["text"]).input_ids
        return b
    return d.map(prep, remove_columns=["audio","text"])
ds_train = to_ds(man["train"]); ds_valc = to_ds(man["valc"]); ds_valr = to_ds(man["valr"])
gc.collect()
print("datasets:", ds_train.num_rows, ds_valc.num_rows, (ds_valr.num_rows if ds_valr else 0))

In [ ]:
# 7. Collator + metrics (raw WER, Greek-normalized WER, CER)
import torch, evaluate, unicodedata, re
from dataclasses import dataclass
@dataclass
class Collator:
    processor: object
    def __call__(self, feats):
        batch = self.processor.feature_extractor.pad([{"input_features": f["input_features"]} for f in feats], return_tensors="pt")
        # Feature extractor emits float32, but the model is loaded in fp16; match dtypes
        # at the batch boundary so encoder conv1 doesn't crash in generate() during eval
        # ("Input type (float) and bias type (c10::Half) should be the same").
        batch["input_features"] = batch["input_features"].to(torch.float16)
        lab = self.processor.tokenizer.pad([{"input_ids": f["labels"]} for f in feats], return_tensors="pt")
        labels = lab["input_ids"].masked_fill(lab.attention_mask.ne(1), -100)
        if (labels[:,0] == self.processor.tokenizer.bos_token_id).all().cpu().item(): labels = labels[:,1:]
        batch["labels"] = labels; return batch
collator = Collator(processor)
wer_m, cer_m = evaluate.load("wer"), evaluate.load("cer")
_PUNCT = re.compile(r"[^\w\s]", re.UNICODE)
def gnorm(s):
    s = unicodedata.normalize("NFD", s.lower()); s = "".join(c for c in s if unicodedata.category(c)!="Mn")
    s = unicodedata.normalize("NFC", s).replace("ς","σ"); return re.sub(r"\s+"," ", _PUNCT.sub(" ", s)).strip()
def metrics(pred):
    lab = np.where(pred.label_ids != -100, pred.label_ids, processor.tokenizer.pad_token_id)
    P = processor.tokenizer.batch_decode(pred.predictions, skip_special_tokens=True)
    R = processor.tokenizer.batch_decode(lab, skip_special_tokens=True)
    return {"wer": 100*wer_m.compute(predictions=P, references=R),
            "wer_norm": 100*wer_m.compute(predictions=[gnorm(x) for x in P], references=[gnorm(x) for x in R]),
            "cer": 100*cer_m.compute(predictions=P, references=R)}

In [ ]:
# 8. Model + LoRA (freeze encoder) + trainer
import torch
from transformers import WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from peft import LoraConfig, get_peft_model
model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.config.suppress_tokens = []
model.generation_config.language, model.generation_config.task = LANGUAGE, TASK
model.model.encoder.requires_grad_(False)
model.gradient_checkpointing_enable(); model.config.use_cache = False
model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                         target_modules=["q_proj","v_proj"]))
model.print_trainable_parameters()
args = Seq2SeqTrainingArguments(output_dir=OUT_DIR, per_device_train_batch_size=TRAIN_BS,
    gradient_accumulation_steps=GRAD_ACC, learning_rate=LR, warmup_ratio=0.1, num_train_epochs=EPOCHS,
    fp16=True, predict_with_generate=True, generation_max_length=225, eval_strategy="epoch",
    save_strategy="epoch", logging_steps=20, report_to=[], remove_unused_columns=False,
    label_names=["labels"], load_best_model_at_end=True, metric_for_best_model="wer", greater_is_better=False,
    seed=SEED, per_device_eval_batch_size=EVAL_BS)
trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_valc,
                         data_collator=collator, compute_metrics=metrics, processing_class=processor)

In [ ]:
# 9. Baseline -> train -> after (val_corr improved? val_reg regressed?)
print("BASELINE val_corr:", trainer.evaluate(ds_valc))
if ds_valr: print("BASELINE val_reg:", trainer.evaluate(ds_valr))
trainer.train()
print("AFTER val_corr:", trainer.evaluate(ds_valc))
if ds_valr: print("AFTER val_reg (regression check):", trainer.evaluate(ds_valr))

In [ ]:
# 10. Save adapter (persisted as Kaggle output on Commit)
import json
model.save_pretrained(OUT_DIR); processor.save_pretrained(OUT_DIR)
json.dump({"model": MODEL_ID, "lora_r": LORA_R, "lr": LR, "epochs": EPOCHS, "seed": SEED,
           "n_val_corr": ds_valc.num_rows, "n_val_reg": (ds_valr.num_rows if ds_valr else 0),
           "val_cities": sorted(VAL_CITIES)}, open(OUT_DIR+"/run_meta.json","w"), ensure_ascii=False, indent=2)
print("saved ->", OUT_DIR)

## N-best / confidence (optional)

Run this after the main pipeline. faster-whisper reports an `avg_logprob` and `no_speech_prob` per segment, plus a `probability` per word. The idea is to emit condensed JSON only for the low-confidence spans, so a downstream LLM only has to fix the parts the model was unsure about. For real N-best you would need beam search with `num_return_sequences`.

In [ ]:
# 11. Confidence-gated condensed output
# Free the fine-tune model FIRST — faster-whisper loads a SECOND large-v3 into
# CTranslate2, so without releasing VRAM this OOMs on a 16GB T4. The LoRA adapter
# was already saved in cell 10, so dropping the trainer/model here is safe.
import gc, torch
for _v in ("trainer", "model"):
    if _v in globals(): del globals()[_v]
gc.collect(); torch.cuda.empty_cache()
from faster_whisper import WhisperModel
CONF_AVG_LOGPROB, CONF_WORD_P = -1.0, 0.7
fw = WhisperModel("large-v3", device="cuda", compute_type="float16")
def annotate(wav_path):
    segs, _ = fw.transcribe(wav_path, language="el", beam_size=5, word_timestamps=True)
    out = []
    for s in segs:
        low = (s.avg_logprob < CONF_AVG_LOGPROB) or any((w.probability or 1) < CONF_WORD_P for w in (s.words or []))
        out.append({"text": s.text.strip(), "conf": round(float(s.avg_logprob),2),
                    "low_words": [w.word for w in (s.words or []) if (w.probability or 1) < CONF_WORD_P]} if low else s.text.strip())
    return out
print("annotate() ready")

## Next

- Compare BASELINE vs AFTER on val_corr (did the corrections improve?) and on val_reg (did anything regress?). The regression check is the one that matters most.
- Scale up: fold the no-edit backbone into training and run a sweep on the mini PC (Track 2).
- Before trusting any number: at least 3 seeds, report the mean and range, and bootstrap confidence intervals by meeting.